# Trainer

In [ ]:
import copy
from sklearn.metrics import hamming_loss, f1_score
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


class MultiLabelTrainer:
    """Trains and evaluates a multi-label model, following the ModelTrainer
    pattern from 3_Multilabel_classification.ipynb but adapted for
    BCEWithLogitsLoss and multi-label metrics."""

    def __init__(
        self,
        model_class: type,
        model_params: dict,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        lr: float = 1e-3,
        num_epochs: int = 100,
    ):
        self.model = model_class(**model_params).to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.lr = lr
        self.num_epochs = num_epochs

        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

        self.best_state = None
        self.history = {"train_loss": [], "val_loss": []}

    # ------ private helpers --------------------------------------------------

    def _train_one_epoch(self):
        self.model.train()
        running_loss = 0.0
        for X_b, y_b in self.train_loader:
            X_b, y_b = X_b.to(self.device), y_b.to(self.device)
            self.optimizer.zero_grad()
            loss = self.criterion(self.model(X_b), y_b)
            loss.backward()
            self.optimizer.step()
            running_loss += loss.item()
        return running_loss / len(self.train_loader)

    @torch.no_grad()
    def _calculate_val_loss(self):
        self.model.eval()
        running_loss = 0.0
        for X_b, y_b in self.val_loader:
            X_b, y_b = X_b.to(self.device), y_b.to(self.device)
            running_loss += self.criterion(self.model(X_b), y_b).item()
        return running_loss / len(self.val_loader)

    # ------ public API -------------------------------------------------------

    def train(self) -> dict:
        best_val_loss = float("inf")

        for epoch in range(self.num_epochs):
            train_loss = self._train_one_epoch()
            val_loss = self._calculate_val_loss()

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                self.best_state = copy.deepcopy(self.model.state_dict())

        self.model.load_state_dict(self.best_state)
        print(f"Training done — best val loss: {best_val_loss:.4f}")
        return self.history

    @torch.no_grad()
    def evaluate(self, loader: DataLoader) -> dict:
        """Compute loss and multi-label metrics on an arbitrary loader."""
        self.model.eval()
        all_logits, all_targets = [], []

        running_loss = 0.0
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(self.device), y_b.to(self.device)
            logits = self.model(X_b)
            running_loss += self.criterion(logits, y_b).item()
            all_logits.append(logits.cpu())
            all_targets.append(y_b.cpu())

        logits = torch.cat(all_logits)
        targets = torch.cat(all_targets).numpy().astype(int)
        preds = (torch.sigmoid(logits) > 0.5).numpy().astype(int)

        return {
            "loss":         running_loss / len(loader),
            "hamming_loss": hamming_loss(targets, preds),
            "micro_f1":     f1_score(targets, preds, average="micro", zero_division=0),
            "macro_f1":     f1_score(targets, preds, average="macro", zero_division=0),
        }

    @staticmethod
    def plot_loss(history: dict, title: str = ""):
        plt.figure(figsize=(8, 4))
        plt.plot(history["train_loss"], label="Train")
        plt.plot(history["val_loss"],   label="Val")
        plt.xlabel("Epoch")
        plt.ylabel("BCEWithLogitsLoss")
        plt.title(f"Loss curve — {title}" if title else "Loss curve")
        plt.legend()
        plt.tight_layout()
        plt.show()

# Data